# Model Building

Prepared by Afees Iyiola O. (UB 24047789)

Trains four classifiers on the prepared training set and records their predictive performance and fairness on the held-out test set. These results form the baseline that the reweighing and equalised odds notebooks are compared against.

The train and test sets are read exactly as the data preparation notebook wrote them. No further splitting or resampling happens here.


## Loading the prepared data

Reads both sets, separates the features from the target and the protected columns, and computes the class weights that counter the attrition imbalance during training.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

RANDOM_SEED = 42
PROTECTED = ["gender_priv", "age_priv", "marital_priv"]

train = pd.read_csv("data/processed/train_dataset.csv")
test = pd.read_csv("data/processed/test_dataset.csv")

X_train = train.drop(columns=PROTECTED + ["Attrition"])
y_train = train["Attrition"]
prot_train = train[PROTECTED]

X_test = test.drop(columns=PROTECTED + ["Attrition"])
y_test = test["Attrition"]
prot_test = test[PROTECTED]

counts = y_train.value_counts()
class_weights = {int(c): len(y_train) / (2 * n) for c, n in counts.items()}

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Feature columns match:", list(X_train.columns) == list(X_test.columns))
print("Training labels: %d retained, %d left" % (counts[0], counts[1]))
print("Class weights:", {k: round(v, 3) for k, v in class_weights.items()})


Train: (1176, 44)  Test: (294, 44)
Feature columns match: True
Training labels: 986 retained, 190 left
Class weights: {0: 0.596, 1: 3.095}


The training set holds 1,176 employees across 44 features, 986 retained against 190 who left, giving class weights of 0.596 and 3.095 so each attrition case counts about five times as much during fitting.

## Training the models

Defines the four classifiers with class weighting applied and evaluates each using 10-fold stratified cross-validation on the training set only. The test set is not touched here.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

scale_pos_weight = counts[0] / counts[1]

models = {
    "Logistic Regression": LogisticRegression(
        class_weight=class_weights, max_iter=1000, random_state=RANDOM_SEED),
    "Decision Tree": DecisionTreeClassifier(
        class_weight=class_weights, random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight=class_weights, random_state=RANDOM_SEED, n_jobs=-1),
    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight, random_state=RANDOM_SEED,
        eval_metric="logloss", n_jobs=-1),
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "pr_auc": "average_precision",
    "brier": "neg_brier_score",
}

cv_results = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"model": name}
    for metric in scoring:
        value = scores["test_" + metric].mean()
        # neg_brier_score is returned negated so that higher is better; flip it back.
        row[metric] = round(-value, 4) if metric == "brier" else round(value, 4)
    cv_results.append(row)

cv_results = pd.DataFrame(cv_results)
print(cv_results.to_string(index=False))


              model  accuracy  precision  recall     f1  roc_auc  balanced_accuracy  pr_auc  brier
Logistic Regression    0.7466     0.3627  0.7421 0.4856   0.8301             0.7448  0.6177 0.1583
      Decision Tree    0.7687     0.2857  0.2789 0.2794   0.5710             0.5710  0.2041 0.2313
      Random Forest    0.8640     0.6399  0.3526 0.4446   0.7988             0.6575  0.5444 0.1170
            XGBoost    0.8682     0.6457  0.4368 0.5154   0.8039             0.6941  0.5657 0.1098


Accuracy inverts the ranking: Decision Tree scores 0.7687 against Logistic Regression's 0.7466 while catching 53 of the 190 who left compared with 141. Logistic Regression leads on recall, ROC-AUC and PR-AUC, but XGBoost has the best Brier score at 0.1098 against 0.1583, so its predicted probabilities are the better calibrated.

## Fitting the final models

Fits each model once on the full training set and predicts the held-out test set. Those predictions are what every performance and fairness measurement is computed from.


In [6]:
from sklearn.metrics import balanced_accuracy_score, recall_score, precision_score
import joblib

MODELS_DIR = Path("results/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

predictions = pd.DataFrame({"y_true": y_test})
for col in PROTECTED:
    predictions[col] = prot_test[col]

test_summary = []
for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    slug = name.lower().replace(" ", "_")
    predictions[slug + "_pred"] = y_pred
    predictions[slug + "_prob"] = y_prob
    joblib.dump(model, MODELS_DIR / f"{slug}_baseline.joblib")



predictions.to_csv("results/predictions_baseline.csv", index=False)

print("Test set: %d attrition cases of %d" % (y_test.sum(), len(y_test)))
print("Models fitted and saved:", list(models))
print("Predictions written to results/predictions_baseline.csv", predictions.shape)


Test set: 47 attrition cases of 294
Models fitted and saved: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']
Predictions written to results/predictions_baseline.csv (294, 12)


The test set holds 47 attrition cases in 294 rows, so recall shifts by just over two percentage points for every single employee the model catches or misses. All four models are fitted and saved, with their predictions and probabilities written alongside the protected columns.

## Measuring performance and fairness

Computes the eight performance metrics and the three fairness metrics across gender, age and marital status, each with a 95 percent bootstrap confidence interval from 1,000 resamples. The same function is called unchanged by the reweighing and equalised odds notebooks, so the three rounds stay comparable.


In [ ]:
from src.metrics import measure_all

baseline_results = measure_all(predictions, list(models), PROTECTED, n_boot=1000, seed=RANDOM_SEED)

TABLES_DIR = Path("results/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
baseline_results.to_csv(TABLES_DIR / "baseline_results.csv", index=False)

perf = baseline_results[baseline_results["attribute"] == "performance"]
print(perf.pivot(index="model", columns="metric", values="value").round(4).to_string())

fair = baseline_results[baseline_results["attribute"] != "performance"].copy()
fair["formatted"] = fair.apply(
    lambda r: "%+.3f [%+.3f, %+.3f]" % (r["value"], r["ci_low"], r["ci_high"]), axis=1)
print()
print(fair.pivot(index=["model", "attribute"], columns="metric", values="formatted").to_string())
